## Environment Setup & Library Installation

In [ ]:
# Installing gensim for Word2Vec
!pip install gensim

In [ ]:
import random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk, spacy, torch
import seaborn as sns

# NLTK Libraries
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# SKlearn Libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity

# Gensim Library for Word2Vec
from gensim.models import Word2Vec

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)

# Huggingface Library to load Dataset
from datasets import load_dataset

# Evaluate Tokens with Counter
from collections import Counter

#perform test on complete dataset
use_full_ds = False  # True or False

# Configuration prameters for the test
CFG = {
    "seed": 42,
    "sample_size": 2000,
    "test_size": 0.2,
    "max_features": 10000,
    "epochs": 3,
    "batch_size": 8
}

warnings.filterwarnings("ignore")
random.seed(CFG["seed"])

# Downloading NLTK
nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("stopwords")
nltk.download("wordnet")

# spaCY loading for English Small "en_core_web_sm"
NLP = spacy.load("en_core_web_sm")

# Selecting CUDA GPU if available
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("Warning!!! GPU not available. \rKindly select GPU for better performance.")
print("All imports successful.")

# Task 1 — Preprocessing & Exploratory Analysis

## 1.1 Load Dataset + Sampling

In [ ]:
ds = load_dataset("banking77")
df_train = pd.DataFrame(ds["train"])
df_test = pd.DataFrame(ds["test"])
df_full = pd.concat(
    [df_train, df_test],
    ignore_index=True
)

label_names = ds["train"].features["label"].names

intent_list = []
for lbl in df_full["label"]:
    intent_list.append(label_names[lbl])

df_full["intent"] = intent_list
if use_full_ds:
  df = df_full
  print(f"Using Full Dataset: {len(df)}")
else:
  df = df_full.sample(n=CFG["sample_size"], random_state=CFG["seed"])
  print(f"Using Sampled Dataset: {len(df)}")

df.head()

## 1.2 Preprocessing (NLTK + spaCy)

In [ ]:
LEM = WordNetLemmatizer()
STOP = set(stopwords.words("english"))

# Preprocessing function to process NLTK or spaCY based on parameter
def preprocess(txt, mode="nltk"):
    result = ""
    if mode == "nltk":
        tokens = word_tokenize(txt.lower())
        result = [LEM.lemmatize(t)
                  for t in tokens
                    if t.isalpha() and t not in STOP
                ]

    if mode == "spacy":
        doc = NLP(txt)
        result = [t.lemma_
                for t in doc
                  if t.is_alpha and not t.is_stop
                ]

    return result

nltk_tokens = []
spacy_tokens = []

for text in df["text"]:
    nltk_tokens.append(preprocess(text, "nltk"))
    spacy_tokens.append(preprocess(text, "spacy"))

## 1.3 Named Entity Recognition (spaCy)

In [ ]:
def extract_entities(text):
    doc = NLP(text)
    entities = []

    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities

entity_col = []
for text in df["text"]:
    entity_col.append(extract_entities(text))

df["entities"] = entity_col

# NER Distribution Barchart
entity_types = []

for row in df["entities"]:
    for e in row:
        entity_types.append(e[1])

pd.Series(entity_types).value_counts().plot(kind="bar")
plt.title("Named Entity Recognition")
plt.show()

## 1.4 Exploratory Visualisations

In [ ]:
# Token Length Distribution
token_lengths = []

for text in df["text"]:
    tokens = word_tokenize(text)
    token_lengths.append(len(tokens))

df["token_len"] = token_lengths

df["token_len"].hist(bins=30)
plt.title("(a) Token Length Distribution")
plt.show()

# Common Words
from collections import Counter

all_tokens = []

for text in df["text"]:
    tokens = preprocess(text)
    all_tokens.extend(tokens)

common_words = Counter(all_tokens).most_common(20)

words = [w[0] for w in common_words]
counts = [w[1] for w in common_words]

plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("(b) Most Common Words")
plt.show()

# Intent Distribution
df["intent"].value_counts().head(15).plot(kind="bar", figsize=(10,5))
plt.title("(c) Top 15 Intents")
plt.show()

## 1.5. Comparison NLTK and spaCy

In [ ]:
# Calculate average token length
nltk_lengths = []
for row in nltk_tokens:
    nltk_lengths.append(len(row))

spacy_lengths = []
for row in spacy_tokens:
    spacy_lengths.append(len(row))

avg_nltk = np.mean(nltk_lengths)
avg_spacy = np.mean(spacy_lengths)


# Calculate vocabulary size
nltk_vocab = set()
for row in nltk_tokens:
    for token in row:
        nltk_vocab.add(token)

spacy_vocab = set()
for row in spacy_tokens:
    for token in row:
        spacy_vocab.add(token)


# Create comparison table
comparison = pd.DataFrame({
    "Metric": ["Avg Tokens", "Vocabulary Size", "NER", "POS Tagging", "Dependency Parsing", "Speed"],
    "NLTK": [avg_nltk, len(nltk_vocab),"No", "Yes (Seperate Tagger)", "No", "Fast"],
    "spaCy": [avg_spacy, len(spacy_vocab), "Yes", "Yes", "Yes", "Moderate"]
})

comparison

## 1.6. Utterance Length Distribution & Reflection

In [ ]:
utterance_lengths = []

for text in df["text"]:
    tokens = word_tokenize(text)
    utterance_lengths.append(len(tokens))

df["utterance_length"] = utterance_lengths

plt.figure(figsize=(8,5))

plt.hist(df["utterance_length"], bins=25)

plt.title("Utterance Length Distribution")
plt.xlabel("Number of Tokens")
plt.ylabel("Frequency")

plt.show()

length_summary = pd.DataFrame({
    "Metric": [
        "Minimum Length",
        "Maximum Length",
        "Average Length",
        "Median Length"
    ],

    "Value": [
        np.min(df["utterance_length"]),
        np.max(df["utterance_length"]),
        np.mean(df["utterance_length"]),
        np.median(df["utterance_length"])
    ]
})

length_summary

## 1.7. Stopword Experiment (NLTK vs spaCy)

In [ ]:
experiment_results = []

methods = {
    "NLTK": "nltk",
    "spaCy": "spacy"
}

for method_name, method_type in methods.items():
    # WITH Stopwords
    tokens_with_stop = []
    for text in df["text"]:
        if method_type == "nltk":
            raw_tokens = []
            tokens = word_tokenize(text.lower())
            for token in tokens:
                if token.isalpha():
                    raw_tokens.append(token)
            tokens_with_stop.append(raw_tokens)
        elif method_type == "spacy":
            raw_tokens = []
            doc = NLP(text.lower())
            for token in doc:
                if token.is_alpha:
                    raw_tokens.append(token.lemma_)
            tokens_with_stop.append(raw_tokens)

    # WITHOUT Stopwords
    tokens_without_stop = []
    for text in df["text"]:
        tokens_without_stop.append(preprocess(text, method_type))

    # Token Statistics
    with_lengths = []
    without_lengths = []

    for row in tokens_with_stop:
        with_lengths.append(len(row))

    for row in tokens_without_stop:
        without_lengths.append(len(row))

    # Vocabulary Statistics

    vocab_with = set()
    for row in tokens_with_stop:
        for token in row:
            vocab_with.add(token)

    vocab_without = set()
    for row in tokens_without_stop:
        for token in row:
            vocab_without.add(token)

    # Store Results
    experiment_results.append([
        method_name,
        np.mean(with_lengths),
        np.mean(without_lengths),
        len(vocab_with),
        len(vocab_without)
    ])

# FINAL COMPARISON TABLE
comparison = pd.DataFrame(
    experiment_results,
    columns=[
        "Method",
        "Avg Tokens With Stopwords",
        "Avg Tokens Without Stopwords",
        "Vocabulary With Stopwords",
        "Vocabulary Without Stopwords"
    ]
)

print("\nStopword Experiment Results:\n")

display(comparison)

# VISUALISATION
plot_data = comparison.set_index("Method")

plot_data[[
    "Avg Tokens With Stopwords",
    "Avg Tokens Without Stopwords"
]].plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Stopword NLTK vs spaCy Token Distribution")
plt.ylabel("Average Token Count")
plt.xticks(rotation=0)

plt.show()

# TASK 2 — Classical Models

## 2.1 TF-IDF Feature Engineering

In [ ]:
X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CFG["test_size"], random_state=CFG["seed"]
)

vec = TfidfVectorizer(max_features=CFG["max_features"], ngram_range=(1,2))
X_train_v = vec.fit_transform(X_train)
X_test_v = vec.transform(X_test)

print("\nTF-IDF Train Shape:", X_train_v.shape)
print("TF-IDF Test Shape :", X_test_v.shape)

## 2.2 Training of Classical Models

In [ ]:
# create a common function to train certain model
def train_model(model):
    start = time.time()

    model.fit(X_train_v, y_train)
    pred = model.predict(X_test_v)

    acc = round(accuracy_score(y_test, pred) * 100, 2)
    macro_f1 = round(f1_score(y_test, pred, average="macro") * 100, 2)
    weight_f1 = round(f1_score(y_test, pred, average="weighted") * 100, 2)
    return acc, macro_f1, weight_f1, pred, time.time() - start

lr_acc, lr_mf1, lr_wf1, lr_pred, lr_time = train_model(LogisticRegression(max_iter=1000))
nb_acc, nb_mf1, nb_wf1, nb_pred, nb_time = train_model(MultinomialNB())
svm_acc, svm_mf1, svm_wf1, svm_pred, svm_time = train_model(LinearSVC())


# create result for the output from the models training.
results = pd.DataFrame([
    ["Logistic Regression", lr_acc, lr_mf1, lr_wf1, lr_time],
    ["Naive Bayes", nb_acc, nb_mf1, nb_wf1, nb_time],
    ["Linear SVM", svm_acc, svm_mf1, svm_wf1, svm_time]
], columns=["Model", "Accuracy %", "Macro F1 %", "Weight F1 %", "Training Time"])


## 2.3 Evaluation of Classical Models

In [ ]:
display(results)

# Confusion Matrix
cm = confusion_matrix(y_test, lr_pred)

cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8,6))
sns.heatmap(cm_norm, cmap="Blues")
plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 2.4 Misclassification Using SVM

In [ ]:
# Misclassified Examples using "svm_pred"
misclassified = pd.DataFrame({
    "Text": X_test,
    "Actual": y_test,
    "Predicted": svm_pred
})

# show top 5 incorrect predictions
misclassified = misclassified[
    misclassified["Actual"] != misclassified["Predicted"]
]

misclassified.head(5)

# TASK 3 — Semantic Embedding RAG-Based Chatbots

## 3.1 Word2Vec Training (Skip-Gram)

In [ ]:
# Tokenise all utterances for Word2Vec
corpus = []

for text in df["text"]:
    corpus.append(word_tokenize(text.lower()))

# Train Skip-Gram Word2Vec
w2v_model = Word2Vec(
    sentences    = corpus,
    vector_size  = 100,   # dimensionality
    window       = 5,     # context window
    min_count    = 2,     # ignore low-frequency words
    sg           = 1,     # 1=Skip-Gram, 0=CBoW
    epochs       = 20,    # training epochs
    workers      = 4,
    seed         = CFG["seed"]
)

vocab_size = len(w2v_model.wv)
print(f"Word2Vec training complete.")
print(f"Vocabulary size : {vocab_size:,}")
print(f"Vector dimension: {w2v_model.vector_size}")

## 3.2 Semantic Similarity Results

In [ ]:
wv = w2v_model.wv

# Banking-relevant query words
query_words = ['card', 'transfer', 'balance', 'fraud', 'exchange']

output = []
for word in query_words:
    if word in wv:
        similar = wv.most_similar(word, topn=5)

        for similar_word, score in similar:
            output.append({
                'Query Word': word,
                'Similar Word': similar_word,
                'Similarity Score %': round(score, 4) * 100
            })
    else:
        output.append({
            'Query Word': word,
            'Similar Word': 'Not in vocabulary',
            'Similarity Score %': None
        })

# Convert to DataFrame
df_output = pd.DataFrame(output)

# Display in tabular format
display(df_output)

## 3.3 Visualisation Word Embedding (PCA and t-SNE)

In [ ]:
# Select top 100 words
words = list(wv.index_to_key)[:100]

# Get vectors
vectors = np.array([wv[word] for word in words])

# PCA Visualization
pca = PCA(n_components=2, random_state=42)
pca_res = pca.fit_transform(vectors)

plt.figure(figsize=(7, 5))

# Scatter plot
plt.scatter(
    pca_res[:, 0],
    pca_res[:, 1],
    alpha=0.7
)

# Add labels for each datapoint
for i, word in enumerate(words):
    plt.annotate(
        word,
        (pca_res[i, 0], pca_res[i, 1]),
        fontsize=8,
        alpha=0.8
    )

plt.title("(a) PCA Word Embedding Visualization", fontsize=16)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# t-SNE Visualization
perp = min(30, len(vectors) - 1)

tsne = TSNE(
    n_components=2,
    perplexity=perp,
    random_state=42,
    init='pca',
    learning_rate='auto'
)

tsne_res = tsne.fit_transform(vectors)

plt.figure(figsize=(7, 5))

# Scatter plot
plt.scatter(
    tsne_res[:, 0],
    tsne_res[:, 1],
    alpha=0.7
)

# Add labels for datapoints
for i, word in enumerate(words):
    plt.annotate(
        word,
        (tsne_res[i, 0], tsne_res[i, 1]),
        fontsize=8,
        alpha=0.8
    )

plt.title("(b) t-SNE Word Embedding Visualization", fontsize=16)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 3.5 Retrieval-Based Chatbot Demonstration

In [ ]:
# Multiple Queries
queries = [
    "lost my credit card",
    "money transfer failed",
    "account balance issue",
    "foreign exchange rate",
    "fraud transaction detected"
]

# Average Word Vector
def avg_vector(words):
    vecs = []
    for w in words:
        if w in wv:
            vecs.append(wv[w])
    if len(vecs) == 0:
        return None
    return np.mean(vecs, axis=0)

# Function: Retrieve Responses
def retrieve(query, top_n=5):
    q_tokens = word_tokenize(query.lower())
    q_vec = avg_vector(q_tokens)
    if q_vec is None:
        return []
    scores = []
    for text in df["text"]:
        t_tokens = word_tokenize(text.lower())
        t_vec = avg_vector(t_tokens)
        if t_vec is not None:
            # Cosine similarity
            similarity = cosine_similarity(
                [q_vec],
                [t_vec]
            )[0][0]
            scores.append((text, similarity))

    # Sort by similarity
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    return scores[:top_n]

# Store Results
all_results = []

for query in queries:
    results = retrieve(query, top_n=5)
    for rank, (response, score) in enumerate(results, start=1):
        all_results.append({
            "Query": query,
            "Rank": rank,
            "Retrieved Response": response,
            "Similarity Score": round(score * 100, 2)
        })

# Convert to DataFrame
results_df = pd.DataFrame(all_results)

# Display results
display(results_df)

# TASK 4 — Transformer DistilBERT

## 4.1.	Model and Training Configuration

In [ ]:
# --- Fix labels ---
label_unique = sorted(df["label"].unique())

label_map = {}
for i, lbl in enumerate(label_unique):
    label_map[lbl] = i

df["label"] = [label_map[l] for l in df["label"]]

# --- Dataset ---
from datasets import Dataset
dataset = Dataset.from_pandas(df[["text","label"]])
dataset = dataset.train_test_split(test_size=0.2)

# --- Tokenizer ---
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

dataset = dataset.map(tokenize, batched=True)

## 4.2.	Fine-tune model

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

# --- Model ---
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_unique)
)

data_collator = DataCollatorWithPadding(tokenizer)

# Metrics Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {
        "accuracy": accuracy,
        "f1": f1
    }

# Training Arguments
args = TrainingArguments(
    output_dir                  = './results',
    num_train_epochs            = CFG["epochs"],
    per_device_train_batch_size = CFG["batch_size"],
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'accuracy',
    report_to                   = 'none',
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
)

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train and Calculate Training Time
start_time = time.time()
trainer.train()
end_time = time.time()

# Total training time
transformer_time = end_time - start_time

## 4.3 Classification Report

In [ ]:
# Predictions
preds = trainer.predict(dataset["test"])

# Get predicted class labels
y_pred = np.argmax(preds.predictions, axis=1)

# Classification Report
report = classification_report(
    dataset["test"]["label"],
    y_pred,
    output_dict=True
)

# Convert report into DataFrame
report_df = pd.DataFrame(report).transpose()

# Round values for readability
report_df = report_df.round(2)

# Display report table
print("\nClassification Report:\n")
display(report_df)

## 4.5 Classical vs Transfomer Model

In [ ]:
transformer_acc = (accuracy_score(dataset["test"]["label"], y_pred) * 100)
transformer_mf1 = (f1_score(dataset["test"]["label"], y_pred, average="macro") * 100)
transformer_wf1 = (f1_score(dataset["test"]["label"], y_pred, average="weighted") * 100)

# Display Comparison Table
compare = pd.DataFrame([
    ["Logistic Regression", round(lr_acc, 2), round(lr_mf1, 2), round(lr_wf1, 2), round(lr_time, 2), "Low" ],
    ["Naive Bayes", round(nb_acc, 2), round(nb_mf1, 2), round(nb_wf1, 2), round(nb_time, 2), "Very Low"],
    ["Linear SVM", round(svm_acc, 2), round(svm_mf1, 2), round(svm_wf1, 2), round(svm_time, 2), "Medium"],
    ["DistilBERT Transformer", round(transformer_acc, 2), round(transformer_mf1, 2), round(transformer_wf1, 2), round(transformer_time, 2), "High"]
], columns=[
    "Model", "Accuracy %", "Macro F1 %", "Weighted F1 %", "Training Time (sec)" , "Model Complexity"
])

print("\nModel Performance Comparison:\n")
display(compare)

## 4.6.	Chabot Deployment Considerations (Intent Recognition)

In [ ]:
# Predictions
preds = trainer.predict(dataset["test"])

y_pred = np.argmax(preds.predictions, axis=1)

y_true = dataset["test"]["label"]

# Create dataframe of results
results_df = pd.DataFrame({
    "Text": dataset["test"]["text"],
    "Actual": y_true,
    "Predicted": y_pred
})

# Get incorrect predictions
errors = results_df[results_df["Actual"] != results_df["Predicted"]]

# Display first 10 errors
display(errors.head(10))